In [1]:
import pandas as pd

customers_df = pd.read_json("noahs-customers.jsonl", lines=True)
orders_df = pd.read_json("noahs-orders.jsonl", convert_dates=['ordered', 'shipped'], lines=True)
products_df = pd.read_json("noahs-products.jsonl", lines=True)

SKU IDs:

- `PET` pets
- `COL` collectible
- `DLI` deli
- `BKY` bakery
- `HOM` household goods

In [2]:
customers_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8260 entries, 0 to 8259
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   customerid    8260 non-null   int64  
 1   name          8260 non-null   str    
 2   address       8260 non-null   str    
 3   citystatezip  8260 non-null   str    
 4   birthdate     8260 non-null   str    
 5   phone         8260 non-null   str    
 6   timezone      8260 non-null   str    
 7   lat           8260 non-null   float64
 8   long          8260 non-null   float64
dtypes: float64(2), int64(1), str(6)
memory usage: 580.9 KB


In [3]:
orders_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 213232 entries, 0 to 213231
Data columns (total 6 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   orderid     213232 non-null  int64         
 1   customerid  213232 non-null  int64         
 2   ordered     213232 non-null  datetime64[us]
 3   shipped     213232 non-null  datetime64[us]
 4   items       213232 non-null  object        
 5   total       213232 non-null  float64       
dtypes: datetime64[us](2), float64(1), int64(2), object(1)
memory usage: 9.8+ MB


In [4]:
products_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1278 entries, 0 to 1277
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sku             1278 non-null   str    
 1   desc            1278 non-null   str    
 2   wholesale_cost  1278 non-null   float64
 3   dims_cm         1278 non-null   object 
dtypes: float64(1), object(1), str(2)
memory usage: 40.1+ KB


In [5]:
customers_df.head()

,customerid,name,address,citystatezip,birthdate,phone,timezone,lat,long
0,1001,Jacqueline Alvarez,105N Elizabeth St,"Manhattan, NY 10013",1958-01-23,315-377-5031,America/New_York,40.718170,-73.997468
1,1002,Julie Howell,185-1 Linden St,"Brooklyn, NY 11221",1956-12-03,680-537-8725,America/New_York,40.694261,-73.921665
2,1003,Christopher Ali,174-28 Baisley Blvd,"Jamaica, NY 11434",2001-09-20,315-846-6054,America/New_York,40.689022,-73.773475
3,1004,Christopher Rodriguez,102 Mount Hope Pl,"Bronx, NY 10453",1959-07-10,516-275-2292,America/New_York,40.849389,-73.909157
4,1005,Jeffrey Wilkinson,17 St Marks Pl,"Manhattan, NY 10003",1988-09-08,838-830-6960,America/New_York,40.728045,-73.987117


In [6]:
norm_orders_df = orders_df.explode('items', ignore_index=True)

In [7]:
norm_orders_df = pd.concat([norm_orders_df.drop(columns=['items']), pd.json_normalize(norm_orders_df['items'])], axis=1)

In [8]:
norm_orders_df.head()

,orderid,customerid,ordered,shipped,total,sku,qty,unit_price
0,1001,6878,2017-01-31 02:56:45,2017-01-31 09:00:00,0.99,PET4571,1,0.99
1,1002,6375,2017-01-31 04:13:35,2017-01-31 12:15:00,13.59,PET4491,1,1.08
2,1002,6375,2017-01-31 04:13:35,2017-01-31 12:15:00,13.59,TOY7498,1,12.51
3,1003,8045,2017-01-31 04:45:12,2017-01-31 10:45:00,1.23,PET5509,1,1.23
4,1004,5385,2017-01-31 05:49:19,2017-01-31 09:00:00,2.10,PET3929,1,2.10


In [9]:
customers_df['initials'] = customers_df['name'].str.split().map(lambda l: "".join([w[0].upper() for w in l]))

In [13]:
df = norm_orders_df.join(
    customers_df.set_index('customerid'),
    on='customerid',
).join(products_df.set_index('sku'), on='sku')
df[(df['initials'] == 'JP') & (df['ordered'].dt.year == 2017) & (df['desc'].str.contains('Bagel'))]['phone']

12791    332-274-4185
Name: phone, dtype: str